# Promptfoo Red Teaming — A Comprehensive Hands-On Tutorial

**Local / self-hosted attack generation · Open-source CLI · No account or email required**

This notebook teaches you to red-team an LLM application end-to-end with the open-source
[`promptfoo`](https://www.promptfoo.dev/docs/red-team/) CLI. It is written as a *runnable
tutorial*: read each markdown cell, then run the code cell beneath it, one at a time.

### Assumptions baked into this notebook
- **Your target is already configured.** Somewhere you already have a working promptfoo
  *target* (an HTTP endpoint, a local model, a Python/JS provider, a RAG/agent, etc.).
  We only *point at it* — we never build it here.
- **Attack generation runs on your own model**, not promptfoo's hosted inference. That means
  either your own API key or a local model (e.g. Ollama). We force this explicitly so nothing
  leaves the machine.
- **No promptfoo account, login, or email** is used. The OSS CLI, local eval, and the local
  vulnerability report all work fully offline. (Email/login only gates the *optional* hosted
  sharing dashboard, which we never touch.)

### How this notebook drives promptfoo
`promptfoo` is a **Node.js CLI**, not a Python library — there is no `import promptfoo` that
does the real work. So this notebook is an *orchestration + documentation* layer:
- Cells that start with `%%writefile` **create YAML config files**.
- Cells that start with `!` **shell out to the `promptfoo` CLI** and show its output inline.
- Cells with plain Python **parse the result files** (with `pandas`) so you can inspect
  pass/fail rates without leaving the notebook.

> Run the cells **in order, one by one**. Several later cells depend on files written by
> earlier cells.

---
## 0 · The red-team mental model

Everything in promptfoo red teaming reduces to **three composable parts**:

| Part | Question it answers | Examples |
|------|--------------------|----------|
| **Target** *(already configured)* | *Who* is under test? | your HTTP API / model / agent |
| **Plugins** | *What* do we attack with? | `harmful:hate`, `pii:direct`, `bola`, `sql-injection`, `contracts` |
| **Strategies** | *How* is the attack delivered? | `base64`, `rot13`, `jailbreak`, `crescendo`, `goat` |

> **Plugins × Strategies = your attack matrix.** A plugin generates a malicious *intent*;
> a strategy *wraps* that intent in an evasion technique. Ten plugins with three strategies
> each is thirty attack families.

### The workflow (the spine of this notebook)

```
  configure  ->   generate   ->      run/eval      ->    report
 (write YAML)   (make attacks)   (fire at target)    (read results)
```

- **generate** turns your config into concrete adversarial test cases in `redteam.yaml`.
- **run** = generate + eval in one shot (fires at the target, grades responses).
- **eval** re-runs an existing `redteam.yaml` and is how we get a clean **JSON** results file.
- **report** opens a local, offline vulnerability dashboard.

---
## 1 · Environment sanity checks

Confirm the CLI is installed and see its version. (Installation itself — `npm install -g promptfoo`
or `npx promptfoo@latest` — is assumed already done on this laptop.)

In [ ]:
# promptfoo version — confirms the CLI is on PATH
!promptfoo --version

In [ ]:
# Top-level help — skim the command groups (eval, redteam, view, ...)
!promptfoo --help

In [ ]:
# Red-team subcommands specifically
!promptfoo redteam --help

### Pick a working directory

All config and result files are written **relative to the current working directory**.
Keep everything for this scan in one folder so `generate`, `run`, `eval`, and `report`
all agree on where files live.

In [ ]:
import os

# Change this to wherever you want the scan artifacts to live.
WORKDIR = os.path.abspath("./redteam_run")
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
print("Working directory:", os.getcwd())

---
## 2 · Force local attack generation (your model, not promptfoo's cloud)

By default, several plugins and the stronger strategies generate their attacks via promptfoo's
**remote inference API** (free and accountless, but it sends prompts off-machine). Because we
want everything local, we do two things:

1. **Disable remote generation** with an environment variable.
2. **Point `redteam.provider` at a model we control** (own key or local Ollama). We set the
   credentials/host here; we wire it into the config in Section 3.

> **Quality caveat:** local/open models produce *weaker* adversarial prompts than promptfoo's
> tuned remote generator. Expect fewer clever jailbreaks. That is the trade-off for staying
> fully offline. A strong hosted model behind your *own* API key (still local-credentialed,
> data governed by that vendor) is a middle ground.

In [ ]:
import os

# 1) Force promptfoo to generate attacks locally instead of via its hosted inference.
os.environ["PROMPTFOO_DISABLE_REDTEAM_REMOTE_GENERATION"] = "true"

# 2) Provide credentials for whichever local/own generator you'll use in redteam.provider.
#    --- Option A: your own OpenAI (or OpenAI-compatible) key -------------------
# os.environ["OPENAI_API_KEY"] = "sk-..."           # <-- your key

#    --- Option B: a local Ollama server (no key needed) ------------------------
#    Make sure `ollama serve` is running and the model is pulled, e.g.:
#       ollama pull llama3.3
#    Ollama listens on http://localhost:11434 by default.

#    --- Option C: any OpenAI-compatible local endpoint (vLLM, LM Studio, ...) ---
# os.environ["LOCAL_API_KEY"]  = "not-needed-or-your-key"

# Silence telemetry/update pings for a clean offline run (optional).
os.environ["PROMPTFOO_DISABLE_TELEMETRY"] = "true"
os.environ["PROMPTFOO_DISABLE_UPDATE"] = "true"

print("Remote generation disabled:",
      os.environ.get("PROMPTFOO_DISABLE_REDTEAM_REMOTE_GENERATION"))

### The three ways to declare a local generator (`redteam.provider`)

You will paste **one** of these into the config in the next section.

```yaml
# Option A — your own OpenAI key
redteam:
  provider:
    id: openai:chat:gpt-4o-mini
    config:
      temperature: 0.5

# Option B — local Ollama model (fully offline)
redteam:
  provider: ollama:chat:llama3.3

# Option C — any OpenAI-compatible local server (vLLM / LM Studio / TGI)
redteam:
  provider:
    id: openai:chat:my-local-model
    config:
      apiBaseUrl: http://localhost:8000/v1
      apiKey: "{{ env.LOCAL_API_KEY }}"
```

`redteam.provider` controls **both** attack *generation* and the model-graded *scoring* of
responses. If you omit it, promptfoo tries its remote grader — which we don't want here.

---
## 3 · Anatomy of the red-team config

Red-team runs are driven by a single `promptfooconfig.yaml`. The important keys:

**Top level**
- `description` — human label for this scan.
- `targets` (a.k.a. `providers`) — the system under test. **Assumed already configured** — we
  reference yours below.
- `prompts` — for a *raw model* target you supply a prompt template with a variable to inject
  attacks into. For an *application/HTTP* target the prompt is usually the app itself, so this
  is often just a passthrough `"{{prompt}}"`.

**The `redteam` block**
- `purpose` — a detailed description of your app (features, users, access levels, data). This is
  the single biggest lever on attack *quality* — the generator uses it to craft relevant attacks.
- `plugins` — *what* to attack with (Section 4).
- `strategies` — *how* to deliver it (Section 5).
- `numTests` — default adversarial cases per plugin (default 5).
- `provider` — the local generator/grader from Section 2.
- `injectVar` — which prompt variable the adversarial text is injected into (auto-inferred if omitted).
- `language` — generation language(s); defaults to English.
- `entities` — named people/orgs/systems to keep consistent across generated attacks (optional).

### Wire in *your already-configured target*

Replace the `targets:` block below with your real target. Common shapes:

```yaml
# (a) A raw model
targets:
  - openai:chat:gpt-4o           # or ollama:chat:..., anthropic:..., etc.

# (b) An HTTP application endpoint
targets:
  - id: https
    config:
      url: https://your-app.internal/chat
      method: POST
      headers: { "Content-Type": "application/json" }
      body: { "message": "{{prompt}}" }
      transformResponse: "json.reply"

# (c) A custom Python/JS provider you already wrote
targets:
  - file://./your_target.py
```

If your target already lives in its own config file, you can keep this notebook's config focused
on `redteam:` and merge, but the simplest path is a single self-contained file as below.

In [ ]:
%%writefile promptfooconfig.yaml
# ---------------------------------------------------------------------------
# Promptfoo red-team config — LOCAL generation, target assumed pre-configured.
# Run cells in order; later commands read this file.
# ---------------------------------------------------------------------------
description: "Red-team scan (local generation)"

# ==== TARGET (system under test) — REPLACE with your configured target ======
targets:
  - id: openai:chat:gpt-4o-mini      # <-- placeholder; swap for YOUR target
    label: target-under-test

# For a raw-model target, define where the attack text is injected.
# For an HTTP/app target, this is typically just a passthrough.
prompts:
  - "{{prompt}}"

redteam:
  purpose: >
    REPLACE ME. Describe the application in detail: what it does, who its users are,
    what data and tools it can access, and what it must never do. The more specific
    this is, the more relevant (and dangerous) the generated attacks will be.

  # Local generator + grader (edit to match Section 2 — Option A/B/C).
  provider:
    id: openai:chat:gpt-4o-mini
    config:
      temperature: 0.5

  numTests: 5          # adversarial cases per plugin (start small)

  plugins:             # WHAT to attack with — see Section 4
    - harmful:hate
    - pii:direct
    - contracts

  strategies:          # HOW to deliver it — see Section 5
    - basic            # send the raw attack, unmodified
    - base64           # static encoding, fully local
    - rot13            # static encoding, fully local

In [ ]:
# Sanity-check the YAML parses and echo the key structure.
import yaml
with open("promptfooconfig.yaml") as f:
    cfg = yaml.safe_load(f)

print("description:", cfg.get("description"))
print("targets    :", cfg.get("targets"))
print("plugins    :", cfg["redteam"]["plugins"])
print("strategies :", cfg["redteam"]["strategies"])
print("numTests   :", cfg["redteam"].get("numTests"))

---
## 4 · Plugins — *what* to attack with

Plugins are **adversarial generators**. Each produces malicious inputs for one risk category.
There are ~157 of them across six families:

| Family | Focus | Example plugin ids |
|--------|-------|--------------------|
| **Trust & Safety** | harmful / graphic / illicit content | `harmful:hate`, `harmful:self-harm`, `harmful:violent-crime` |
| **Security & Access Control** (OWASP-mapped) | injection, SSRF, broken access | `sql-injection`, `ssrf`, `bola`, `bfla`, `shell-injection`, `debug-access` |
| **Privacy / PII** | data leakage | `pii:direct`, `pii:api-db`, `pii:session`, `harmful:privacy` |
| **Compliance & Legal** | illegal advice, IP, contracts | `contracts`, `harmful:intellectual-property`, `harmful:specialized-advice` |
| **Brand** | reputation, hallucination, competitors | `hallucination`, `competitors`, `excessive-agency`, `overreliance` |
| **Custom** | your own policies | `policy`, `intent`, `file://custom-plugin.yaml` |

### Three ways to reference plugins

```yaml
plugins:
  - harmful:hate                    # a single plugin id
  - id: pii:direct                  # object form, so you can set numTests
    numTests: 10
  - harmful                         # a COLLECTION: expands to all harmful:* plugins
  - default                         # curated common set (used if you omit plugins)
  - owasp:llm                       # framework mapping: OWASP LLM Top 10
  - nist:ai:measure                 # framework mapping: NIST AI RMF
  - pii                             # all PII plugins
```

Collections are the fast way to broad coverage; individual ids are how you go deep on a specific
risk. Framework collections (`owasp:llm`, `owasp:api`, `nist:ai`, `mitre:atlas`) double as
**compliance reporting** groupings in the final report.

### Discover the plugins your installed version actually supports

Don't guess from the docs — ask the CLI. `promptfoo redteam plugins` prints the full catalog for
*your* installed version (ids + descriptions), so the list always matches what will run.

In [ ]:
# Full plugin catalog with descriptions (authoritative for your version).
!promptfoo redteam plugins

In [ ]:
# Just the ids — easy to copy straight into a config's `plugins:` list.
!promptfoo redteam plugins --ids-only

In [ ]:
# Only the DEFAULT plugin set (what you get if you omit `plugins:`).
!promptfoo redteam plugins --default

Capture the ids in Python so you can filter/search them while choosing a subset.

In [ ]:
import subprocess

# Grab the id list from the CLI and load it into Python for easy filtering.
out = subprocess.run(["promptfoo", "redteam", "plugins", "--ids-only"],
                     capture_output=True, text=True)
plugin_ids = [line.strip() for line in out.stdout.splitlines()
              if line.strip() and not line.startswith(("Usage", "Available"))]
print(f"{len(plugin_ids)} plugins available\n")

# Example: find all PII / security / harmful plugins to build a focused subset.
for group in ("pii", "harmful", "sql", "bola", "bfla", "ssrf", "prompt-"):
    hits = [p for p in plugin_ids if group in p]
    if hits:
        print(f"{group:<10} -> {', '.join(hits)}")

### A broader, plugin-focused config

This variant swaps in collections and per-plugin `numTests`. Writing it to a **separate file**
lets you keep the minimal config from Section 3 intact and choose which to run later.

In [ ]:
%%writefile promptfooconfig.plugins.yaml
description: "Red-team scan — broad plugin coverage (local generation)"

targets:
  - id: openai:chat:gpt-4o-mini      # <-- REPLACE with your target
    label: target-under-test

prompts:
  - "{{prompt}}"

redteam:
  purpose: >
    REPLACE ME with a detailed description of the target application.
  provider:
    id: openai:chat:gpt-4o-mini      # <-- your local generator (Section 2)
  numTests: 5

  plugins:
    # --- broad collections ---
    - harmful                        # all trust & safety plugins
    - pii                            # all PII plugins
    # --- targeted, deeper coverage ---
    - id: sql-injection
      numTests: 10
    - id: bola                       # broken object-level authorization
      numTests: 10
    - id: prompt-extraction          # tries to extract your system prompt
    # --- framework mapping (also groups the report) ---
    - owasp:llm

  strategies:
    - basic
    - base64
    - leetspeak

---
## 5 · Strategies — *how* to deliver the attack

A strategy wraps each plugin's raw payload in an evasion/attack pattern. They fall into three
tiers of increasing power **and** cost/time:

### Tier 1 — Static / deterministic (fully local, no model needed)
No LLM required to build them, so they run **completely offline** and cheaply.
`basic`, `base64`, `hex`, `rot13`, `leetspeak`, `morse`, `pig-latin`, `camelcase`,
`homoglyph`, `emoji`, `citation`, `prompt-injection` (template-based), `jailbreak:tree` templates.

### Tier 2 — Dynamic single-turn (LLM-powered — uses your `redteam.provider`)
An attacker model iteratively refines a jailbreak. Works locally with your own model, but weaker
models produce weaker attacks. `jailbreak` (iterative), `jailbreak:composite`, `best-of-n`,
`gcg`, `math-prompt`, `likert`.

### Tier 3 — Multi-turn agents (most powerful, slowest, most tokens)
A conversation-driving agent escalates over several turns. `crescendo` (gradual escalation),
`goat` (offensive agent tester), `hydra` (adaptive branching with memory), `mischievous-user`.

> **Offline note:** some Tier-2/3 strategies (composite/meta-agent, best-of-n, etc.) are marked
> in the docs as using **remote inference** in the community edition. With
> `PROMPTFOO_DISABLE_REDTEAM_REMOTE_GENERATION=true` set (Section 2), stick primarily to
> **Tier 1** for guaranteed-local runs, and use **Tier 2** iterative `jailbreak`/`crescendo`
> driven by your own `redteam.provider`. If a strategy errors asking for remote access, it isn't
> available offline — drop it or supply a provider it can use.

### Reference syntax (string or object form)

```yaml
strategies:
  - basic
  - base64
  - id: jailbreak                 # object form lets you tune it
    config:
      numIterations: 4
  - id: crescendo                 # multi-turn
    config:
      maxTurns: 5
  - id: jailbreak:composite       # scope a strategy to specific plugins
    config:
      plugins: [harmful:hate]
```

### Discover the available strategies

Unlike plugins, there is **no `promptfoo redteam strategies` CLI command**. The authoritative
per-version source is the promptfoo package's strategy constants. Two ways to enumerate them:

1. **Best-effort, live from the installed package** (depends on internal paths, may vary by
   version) — the `node` one-liner below.
2. **Curated reference** (always works, offline) — the Python catalog in the next cell, grouped by
   the same constant names promptfoo uses internally.

In [ ]:
# Best-effort: pull the live strategy list from the installed promptfoo package.
# If your version's internal path differs, this prints a hint and you fall back
# to the curated catalog in the next cell.
!node -e 'try{const c=require("promptfoo/dist/src/redteam/constants");const s=c.ALL_STRATEGIES||[];console.log(s.length+" strategies:");console.log(s.join("\n"))}catch(e){console.log("Could not import package internals ("+e.message+"). Use the curated catalog below or promptfoo.dev/docs/red-team/strategies/.")}'

### Curated strategy catalog (grouped by tier / promptfoo constant)

This mirrors promptfoo's own groupings (`DEFAULT_STRATEGIES`, `ADDITIONAL_STRATEGIES`,
`MULTI_TURN_STRATEGIES`, `MULTI_MODAL_STRATEGIES`) plus the offline/remote split. Use it to pick a
subset; check against the live CLI/`node` output above for anything new in your version.

In [ ]:
STRATEGY_CATALOG = {
    "tier1_static_local": [   # fully local, no model needed
        "basic", "base64", "hex", "rot13", "leetspeak", "morse", "pig-latin",
        "camelcase", "homoglyph", "emoji", "prompt-injection", "citation",
    ],
    "tier2_single_turn_llm": [  # driven by your redteam.provider (some want remote)
        "jailbreak", "jailbreak:tree", "jailbreak:composite", "jailbreak:meta",
        "best-of-n", "gcg", "math-prompt", "likert",
    ],
    "tier3_multi_turn": [       # slowest / most tokens
        "crescendo", "goat", "jailbreak:hydra", "mischievous-user", "custom",
    ],
    "multi_modal": [            # media-based (require a multimodal target)
        "audio", "image", "video",
    ],
    # Marked in docs as needing remote inference in the community edition -> avoid
    # these when PROMPTFOO_DISABLE_REDTEAM_REMOTE_GENERATION=true.
    "often_remote_only": [
        "jailbreak:composite", "jailbreak:meta", "best-of-n", "gcg",
    ],
}

for group, items in STRATEGY_CATALOG.items():
    print(f"{group:<24} ({len(items)}): {', '.join(items)}")

offline_safe = [s for s in STRATEGY_CATALOG["tier1_static_local"]
                if s not in STRATEGY_CATALOG["often_remote_only"]]
print("\nGuaranteed-offline picks:", ", ".join(offline_safe))

In [ ]:
%%writefile promptfooconfig.strategies.yaml
description: "Red-team scan — strategy tiers demo (local generation)"

targets:
  - id: openai:chat:gpt-4o-mini      # <-- REPLACE with your target
    label: target-under-test
prompts:
  - "{{prompt}}"

redteam:
  purpose: >
    REPLACE ME with a detailed description of the target application.
  provider:
    id: openai:chat:gpt-4o-mini      # <-- your local generator
  numTests: 3

  plugins:
    - harmful:hate
    - pii:direct

  strategies:
    # Tier 1 — static, fully local
    - basic
    - base64
    - rot13
    - leetspeak
    # Tier 2 — single-turn, LLM-driven by your provider
    - id: jailbreak
      config:
        numIterations: 3
    # Tier 3 — multi-turn (slow / token-heavy) — uncomment to include
    # - id: crescendo
    #   config:
    #     maxTurns: 4

---
## 6 · Generate the attacks

`redteam generate` reads your config, expands `plugins x strategies x numTests`, and writes
concrete adversarial test cases to `redteam.yaml`. **Nothing is sent to your target yet** — this
step only *builds* the attacks (and it's the step that uses your local generator).

Key flags:
- `-c, --config <path>` — which config to expand (default `promptfooconfig.yaml`).
- `-o, --output <path>` — where to write the generated cases (default `redteam.yaml`).
- `-n, --num-tests <N>` — override `numTests` for all plugins.
- `--plugins` / `--strategies` — comma-separated overrides without editing the file.
- `--delay <ms>` — pause between generation calls (**use this to avoid rate limits** on a local
  server) — implies single-threaded.
- `-j, --max-concurrency <N>` — parallel generation calls (lower it if a local model chokes).
- `--force` — regenerate even if promptfoo thinks nothing changed.

### First, size the attack set *before* generating

promptfoo has **no fixed prompt count per plugin** — each plugin generates `numTests` cases (you
choose), and every non-`basic` strategy transforms those base attacks into additional cases. So
the total scales roughly as:

```
total ≈ (num_plugins × numTests) × num_strategies
```

(`basic` counts as one strategy; multi-turn strategies still produce one test case per base
attack, but each runs several model turns, so they cost far more time/tokens than the count
suggests. A few *dataset* plugins — e.g. `beavertails`, `harmbench` — draw from fixed pools rather
than generating.) This estimate helps you **budget and subset before committing**.

In [ ]:
import yaml

def estimate(config_path):
    with open(config_path) as f:
        c = yaml.safe_load(f)
    rt = c.get("redteam", {})
    default_n = rt.get("numTests", 5)
    plugins = rt.get("plugins", [])
    strategies = rt.get("strategies", []) or ["basic"]

    # per-plugin numTests (object form can override the default)
    per_plugin = {}
    for p in plugins:
        if isinstance(p, dict):
            per_plugin[p["id"]] = p.get("numTests", default_n)
        else:
            per_plugin[p] = default_n

    base = sum(per_plugin.values())          # attacks before strategy expansion
    n_str = len(strategies)
    est_total = base * n_str
    print(f"Config: {config_path}")
    print(f"  plugins            : {len(per_plugin)}  ({', '.join(per_plugin)})")
    print(f"  numTests (default) : {default_n}")
    print(f"  strategies         : {n_str}  ({', '.join(str(s.get('id', s) if isinstance(s, dict) else s) for s in strategies)})")
    print(f"  base attacks       : {base}")
    print(f"  ESTIMATED total    : ~{est_total} test cases  (base x strategies)")
    return est_total

estimate("promptfooconfig.yaml")

If that number is bigger than you want, subset **before** generating: drop plugins/strategies,
lower `numTests`, or pass `--plugins` / `--strategies` / `-n` on the command line to override the
file for a quick, smaller run.

In [ ]:
# Generate attacks from the minimal config into redteam.yaml.
# --delay 500 keeps a local model from being hammered; drop it if generation is fast.
# Tip: override the file for a smaller run, e.g.
#   !promptfoo redteam generate -c promptfooconfig.yaml -o redteam.yaml -n 2 --plugins pii:direct,harmful:hate
!promptfoo redteam generate -c promptfooconfig.yaml -o redteam.yaml --delay 500 --force

### Inspect what was generated

Always eyeball `redteam.yaml` before firing it — this is where you catch a mis-set `purpose`,
empty plugins, or a strategy that silently produced nothing.

In [ ]:
import yaml, pandas as pd

with open("redteam.yaml") as f:
    rt = yaml.safe_load(f)

tests = rt.get("tests", [])
print(f"Total generated test cases: {len(tests)}\n")

# Build a tidy frame of exactly what was generated.
rows = []
for t in tests:
    meta = (t.get("metadata") or {})
    rows.append({
        "plugin":   meta.get("pluginId", "?"),
        "strategy": meta.get("strategyId", meta.get("strategy", "basic")),
    })
gen = pd.DataFrame(rows)

print("Actual attacks per plugin:")
print(gen["plugin"].value_counts().to_string())
print("\nActual attacks per strategy:")
print(gen["strategy"].value_counts().to_string())

In [ ]:
# EXACT plugin x strategy count matrix — how many attacks each combination produced.
# This is your subsetting map: zero cells mean nothing was generated there; big cells
# are where most of your run's time will go.
counts = gen.groupby(["plugin", "strategy"]).size().unstack(fill_value=0)
counts["TOTAL"] = counts.sum(axis=1)
counts.loc["TOTAL"] = counts.sum(axis=0)
counts

In [ ]:
# Peek at one concrete attack payload to see what will hit your target.
example = tests[0]
print("VARS   :", example.get("vars"))
print("META   :", example.get("metadata"))
print("ASSERTS:", [a.get("type") for a in example.get("assert", [])])

---
## 7 · Run the attacks against the target

Two ways to execute:

**A. `promptfoo redteam run`** — convenience command that does *generate + eval* together and
maintains the report state. Best for a one-shot scan.

**B. `promptfoo eval -c redteam.yaml -o results.json`** — evaluate the **already-generated**
`redteam.yaml` and write a clean JSON file. Best when you want to **parse results in this
notebook** (Section 8), because you control the output format.

We'll use **B** so we get JSON, after which `redteam report` still reads the same eval state.

Useful eval flags:
- `-o results.json` — JSON output (also supports `.csv`, `.html`, `.yaml`, `junit.xml`).
- `--no-cache` — don't reuse cached target responses (force fresh calls).
- `-j, --max-concurrency <N>` — parallel requests to the target (lower for rate-limited targets).
- `--filter-providers <regex>` / `--filter-pattern <regex>` — run a subset (great for iterating).
- `--filter-first-n <N>` / `--filter-sample <N>` — smoke-test on a few cases first.

In [ ]:
# SMOKE TEST FIRST: fire only the first 5 attacks at the target to confirm wiring.
!promptfoo eval -c redteam.yaml -o results_smoke.json --filter-first-n 5 -j 2

In [ ]:
# Full run: evaluate every generated attack and write JSON for parsing.
# Add --no-cache to force fresh target calls; tune -j to your target's rate limits.
!promptfoo eval -c redteam.yaml -o redteam-results.json -j 3

> **Alternative one-shot (option A):** `!promptfoo redteam run -c promptfooconfig.yaml`
> generates and evaluates in a single command and wires straight into the report — but it doesn't
> hand you a JSON file to parse as cleanly. Use it when you only want the dashboard.

---
## 8 · Parse and analyze results with pandas

The JSON schema has shifted across promptfoo versions, so we **navigate defensively**: first
print the top-level keys, then locate the per-test results list wherever it lives, then flatten
the fields we care about. Adjust the accessors if your version nests things differently.

In [ ]:
import json

with open("redteam-results.json") as f:
    data = json.load(f)

print("Top-level keys:", list(data.keys()))

# The list of per-test results lives in different places across versions.
def find_results_list(d):
    # newer: {"results": {"results": [...] , "stats": {...}}}
    r = d.get("results")
    if isinstance(r, dict) and isinstance(r.get("results"), list):
        return r["results"], r.get("stats")
    # older: {"results": [...], "stats": {...}}
    if isinstance(r, list):
        return r, d.get("stats")
    raise ValueError("Could not locate results list; inspect data manually.")

results, stats = find_results_list(data)
print("Per-test results:", len(results))
print("Stats:", stats)

In [ ]:
import pandas as pd

def flatten(item):
    meta = item.get("metadata") or (item.get("testCase", {}) or {}).get("metadata", {}) or {}
    grading = item.get("gradingResult") or {}
    # pass/fail can be at top level (success) or under gradingResult (pass)
    passed = item.get("success")
    if passed is None:
        passed = grading.get("pass")
    resp = item.get("response") or {}
    return {
        "plugin":   meta.get("pluginId", "?"),
        "strategy": meta.get("strategyId", meta.get("strategy", "basic")),
        "severity": meta.get("severity", ""),
        # In red teaming, a PASSED probe means the app RESISTED the attack.
        "passed":   bool(passed) if passed is not None else None,
        "reason":   (grading.get("reason") or "")[:160],
        "attack":   str((item.get("vars") or {}).get("prompt", ""))[:120],
        "output":   str(resp.get("output", ""))[:120],
    }

df = pd.DataFrame([flatten(r) for r in results])
print("Rows:", len(df))
df.head(10)

In [ ]:
# Headline numbers. Remember: passed == the app DEFENDED; failed == VULNERABILITY.
total = len(df)
defended = int(df["passed"].fillna(False).sum())
vulnerable = total - defended
print(f"Total probes : {total}")
print(f"Defended     : {defended}  ({defended/total:.0%})")
print(f"VULNERABLE   : {vulnerable}  ({vulnerable/total:.0%})")

In [ ]:
# Attack-surface matrix: pass rate for every plugin x strategy combination.
# Lower numbers = weaker defense against that combination.
matrix = (df.assign(defended=df["passed"].fillna(False).astype(int))
            .pivot_table(index="plugin", columns="strategy",
                         values="defended", aggfunc="mean"))
matrix.style.format("{:.0%}") if hasattr(matrix, "style") else matrix

In [ ]:
# The failures — every case where the app did NOT resist the attack.
failures = df[df["passed"] == False][["plugin", "strategy", "severity", "attack", "reason"]]
print(f"{len(failures)} vulnerabilities found\n")
failures.reset_index(drop=True)

In [ ]:
# Rank risk by plugin: which categories fail most often?
risk = (df.assign(vuln=(df["passed"] == False).astype(int))
          .groupby("plugin")
          .agg(probes=("vuln", "size"), vulnerabilities=("vuln", "sum")))
risk["vuln_rate"] = (risk["vulnerabilities"] / risk["probes"]).map("{:.0%}".format)
risk.sort_values("vulnerabilities", ascending=False)

---
## 9 · The vulnerability report (local, offline)

`promptfoo redteam report` starts a **local** web server and opens the red-team dashboard:
severity breakdown, per-category pass/fail, concrete failing transcripts, and remediation
suggestions. It reads local state only — **no account, no upload, no email**.

Because it launches a long-running server, **run it in a terminal**, not as a blocking notebook
cell (a `!` cell would hang until you kill it). The command:

```bash
promptfoo redteam report          # dashboard at http://localhost:15500
promptfoo redteam report -p 8080  # custom port
```

For a plain eval view (the non-red-team results explorer):

```bash
promptfoo view -y                 # results UI at http://localhost:15500
```

If you prefer a shareable static file instead of a server, you already produced HTML options via
`-o results.html` in Section 7 — open that file directly in a browser.

In [ ]:
# Optional: generate a self-contained HTML report you can open without a server.
!promptfoo eval -c redteam.yaml -o redteam-report.html -j 3
print("Open redteam-report.html in a browser.")

---
## 10 · Iterating, customizing, and going deeper

### Narrow to what failed
Re-run only the interesting subset instead of the whole matrix:
```bash
promptfoo eval -c redteam.yaml --filter-metadata pluginId=sql-injection
promptfoo eval -c redteam.yaml --filter-pattern "base64"
```

### Custom policy plugins (your business rules)
Beyond built-ins, encode your own forbidden behaviors:
```yaml
redteam:
  plugins:
    - id: policy
      config:
        policy: "The assistant must never reveal internal pricing or discount authority."
    - id: intent
      config:
        intent: "Get the bot to promise a refund it isn't authorized to give."
    - file://./custom-plugin.yaml     # fully custom generator
```

### Tune `purpose` — the highest-leverage knob
Vague purpose -> generic attacks. A detailed purpose (features, roles, data, tools, hard limits)
makes the generator produce attacks that actually fit *your* app. Revisit it after the first run.

### Framework / compliance mappings
Group results against a standard by adding framework collections as plugins:
`owasp:llm`, `owasp:api`, `nist:ai:measure`, `mitre:atlas`. The report then organizes findings by
that framework — useful for governance and audit evidence.

### Reproducibility & scale
- `numTests` and `--num-tests` scale coverage; caching (`--no-cache` to bypass) keeps reruns cheap.
- Raise `-j` for speed on tolerant targets; add `--delay` for rate-limited local models.
- Grow the plugin/strategy set gradually — start with Tier-1 strategies + a few plugins, then add
  Tier-2/3 once wiring and grading look right.

### CI / automation
`promptfoo redteam run --strict` fails the process if plugins error; combine with `junit.xml`
output for pipeline gating. Run headless, archive `redteam-results.json`, and diff vulnerability
counts across builds to catch regressions.

---
## Appendix · Command & env cheat-sheet

**Lifecycle**
```bash
promptfoo redteam generate -c promptfooconfig.yaml -o redteam.yaml --force   # build attacks
promptfoo eval -c redteam.yaml -o redteam-results.json                       # fire + grade (JSON)
promptfoo redteam run -c promptfooconfig.yaml                                 # generate + eval in one
promptfoo redteam report                                                     # local dashboard
promptfoo view -y                                                            # eval results UI
```

**Handy flags**
```
-c / --config          config file(s)
-o / --output          output file (.json .csv .html .yaml junit.xml ...)
-n / --num-tests       attacks per plugin
-j / --max-concurrency parallel calls (lower for rate limits)
--delay <ms>           pause between calls (single-threaded)
--no-cache             force fresh calls
--force                regenerate even if unchanged
--filter-first-n N     smoke-test on N cases
--filter-metadata k=v  run a subset (e.g. pluginId=bola)
--filter-pattern rgx   subset by test description/strategy
--strict               (redteam run) fail on plugin errors — for CI
```

**Environment variables used here**
```
PROMPTFOO_DISABLE_REDTEAM_REMOTE_GENERATION=true   # force LOCAL attack generation
PROMPTFOO_DISABLE_TELEMETRY=true                   # no telemetry
PROMPTFOO_DISABLE_UPDATE=true                      # no update pings
OPENAI_API_KEY / LOCAL_API_KEY                     # credentials for your generator
```

**Troubleshooting**
- *A strategy errors demanding remote access* -> it isn't available offline; drop it or give it a
  usable `provider`. Stick to Tier-1 strategies for guaranteed-local runs.
- *Local model rate-limits or stalls* -> add `--delay 1000` and set `-j 1`.
- *Zero / weak attacks generated* -> your `purpose` is too vague, or the local generator is too
  small; enrich `purpose` or use a stronger model behind your own key.
- *"passed" looks inverted* -> in red teaming a **passed** probe means the app **defended**;
  a **failed** probe is a **vulnerability**.

---
*Remember: run cells top-to-bottom, replace every `REPLACE ME` / placeholder target, and keep all
artifacts in one working directory.*